# BrainMoE-PINN on Google Colab (GPU smoke)


Select a GPU runtime (T4 is sufficient for this bounded smoke). This notebook runs one real `BrainMoETrainer` update with the path-calibrated data-profile objective: empirical CRPS, projected path Energy Score, sampled-path autocorrelation, and H5–H8 overdispersion. The profile disables the ensemble-mean Huber center loss. The batch is synthetic; this verifies runtime and optimizer wiring, not scientific performance.

Before running, create a source archive from the repository root and upload it to `/content/brain_moe_pinn_colab.tar.gz`. With the `colab` CLI, run `colab upload -s <session-name> brain_moe_pinn_colab.tar.gz /content/brain_moe_pinn_colab.tar.gz`; the Colab Files panel also works:

```bash
tar -czf brain_moe_pinn_colab.tar.gz --exclude=.git --exclude=.venv --exclude=.slim --exclude=__pycache__ --exclude=.pytest_cache --exclude='*.pyc' --exclude='*.pt' --exclude='*.pth' .
```

In [ ]:
from pathlib import Path
import sys
import tarfile

archive_path = Path('/content/brain_moe_pinn_colab.tar.gz')
assert archive_path.is_file(), 'Upload brain_moe_pinn_colab.tar.gz to /content first'
project_root = Path('/content/brain_moe_pinn')
project_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(archive_path, 'r:gz') as archive:
    archive.extractall(project_root, filter='data')
assert (project_root / 'train.py').is_file()
sys.path.insert(0, '/content')
print('PROJECT_ROOT', project_root)

In [ ]:
import torch

assert torch.cuda.is_available(), 'Enable a GPU runtime in Runtime > Change runtime type'
print('COLAB_RUNTIME', torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
from torch.utils.data import DataLoader, Dataset

from brain_moe_pinn import BrainMoEPINN
from brain_moe_pinn.training.training_loop import BrainMoETrainer
from brain_moe_pinn.training.training_phases import (
    LossWeights,
    TrainingPhase,
    TrainingStage,
)

class OneBatch(Dataset):
    def __len__(self):
        return 1

    def __getitem__(self, index):
        return {
            'calcium': torch.randn(12, 64),
            'calcium_future': torch.randn(8, 12, 64),
            'dt': torch.tensor(15.0 / 64),
        }

torch.manual_seed(42)
model = BrainMoEPINN(
    eeg_channels=2,
    fmri_regions=3,
    latent_dim=8,
    use_neurostorm=False,
    use_kda_decoder=False,
    use_active_inference=False,
    use_species_conditioning=True,
    species='c_elegans',
    species_vocab=['c_elegans', 'human'],
    perturbation_dim=2,
    moe_num_shared=1,
    moe_num_routed=1,
    moe_top_k=1,
    generic_observation_only=True,
    transition_mode='sde',
    diffusion_rank=3,
    stochastic_samples=3,
)
head_before = model.transition_diffusion.rank_scale[-1].weight.detach().clone()
loader = DataLoader(OneBatch(), batch_size=1)
trainer = BrainMoETrainer(
    model,
    {
        'model_config': {'species': 'c_elegans'},
        'experiment_data': {
            'modalities': ['calcium'],
            'paired_next_step_targets': True,
            'future_steps': 8,
        },
        'generic_channels': 12,
        'generic_time': 64,
        'use_mixer': False,
    },
    log_dir='/content/brain_moe_pinn_colab_logs',
    checkpoint_dir='/content/brain_moe_pinn_colab_checkpoints',
    train_dataloader=loader,
)
phase = TrainingPhase(
    name='colab_sde_path_calibrated_smoke',
    stage=TrainingStage.STAGE_1_P1,
    total_steps=1,
    learning_rate=5e-5,
    warmup_steps=0,
    batch_size=1,
    rollout_steps=8,
    loss_weights=LossWeights(
        recon_eeg=0.0,
        recon_fmri=0.0,
        recon_meg=0.0,
        forecast=1.0,
        forecast_huber=0.0,
        forecast_corr_diff=0.0,
        forecast_crps=0.25,
        forecast_path_energy=1.0,
        forecast_path_autocorr=0.05,
        forecast_overdispersion=0.1,
        forecast_horizon_weights=(1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3),
        recon_extra={'calcium': 0.0},
    ),
)
runtime_phase = phase.to_runtime_config()
runtime_phase['freeze_policy'] = {
    'trainable_parameter_prefixes': (
        'velocity_brain.mobility_op.',
        'velocity_brain.arousal_vector',
        'velocity_brain.arousal_scale',
        'transition_diffusion.',
    ),
    'trainable_parameter_lr_scales': {
        'velocity_brain.mobility_op.': 0.2,
        'velocity_brain.arousal_vector': 0.2,
        'velocity_brain.arousal_scale': 0.2,
        'transition_diffusion.': 0.02,
    },
}
mobility_before = {
    name: parameter.detach().clone()
    for name, parameter in model.velocity_brain.mobility_op.named_parameters()
}
trainer.train_phase(runtime_phase)
loss_row = trainer.logger.metrics_history[-1]
loss_weights = {
    'forecast_crps': 0.25,
    'forecast_path_energy': 1.0,
    'forecast_path_autocorr': 0.05,
    'forecast_overdispersion': 0.1,
}
loss_metric_names = {
    'forecast_crps': 'forecast_crps',
    'forecast_path_energy': 'forecast_path_energy',
    'forecast_path_autocorr': 'forecast_path_autocorr',
    'forecast_overdispersion': 'forecast_overdispersion',
}
raw_losses = {
    name: float(loss_row[f'loss_{metric_name}_calcium'])
    for name, metric_name in loss_metric_names.items()
}
weighted_losses = {
    name: raw_losses[name] * weight
    for name, weight in loss_weights.items()
}
assert all(torch.isfinite(torch.tensor(value)) for value in raw_losses.values())
print('RAW_LOSS_TERMS', raw_losses)
print('WEIGHTED_LOSS_CONTRIBUTIONS', weighted_losses)
head_after = model.transition_diffusion.rank_scale[-1].weight.detach()
mobility_deltas = [
    (parameter.detach() - mobility_before[name].to(parameter.device)).abs().sum()
    for name, parameter in model.velocity_brain.mobility_op.named_parameters()
]
diffusion_delta = float((head_after - head_before.to(head_after.device)).abs().sum().item())
mobility_delta = float(sum(delta.item() for delta in mobility_deltas))
assert torch.isfinite(head_after).all()
assert all(
    torch.isfinite(parameter).all()
    for parameter in model.velocity_brain.mobility_op.parameters()
)
assert diffusion_delta > 0.0, 'Path-calibrated loss did not update diffusion'
assert mobility_delta > 0.0, 'Path-calibrated loss did not update mobility'
print('COLAB_SDE_PATH_CALIBRATED_TRAIN_STEP_OK', diffusion_delta, mobility_delta)